In [0]:
from pyspark.sql.functions import col, row_number, lag, date_format
from pyspark.sql.window import Window

# ==========================================
# 1. Setup Variables
# ==========================================
catalog = "maritime_ais"
silver_schema = "maritime_silver"
gold_schema = "maritime_gold"

print("Building the Gold Layer...")

# ==========================================
# 2. Query Silver Tables
# ==========================================
df_ais = spark.read.table(f"{catalog}.{silver_schema}.ais_locations")
df_ports = spark.read.table(f"{catalog}.{silver_schema}.port_calls")

# ==========================================
# 3. GOLD TABLE #1 — Current Snapshot (latest state per vessel)
# ==========================================
print("Building current_vessel_status...")

window_spec_ais = Window.partitionBy("mmsi").orderBy(col("timestamp").desc())
df_latest_ais = (
    df_ais.withColumn("rn", row_number().over(window_spec_ais))
    .filter(col("rn") == 1)
    .drop("rn")
)

window_spec_ports = Window.partitionBy("mmsi").orderBy(col("call_timestamp").desc())
df_latest_ports = (
    df_ports.withColumn("rn", row_number().over(window_spec_ports))
    .filter(col("rn") == 1)
    .drop("rn")
)

df_gold_current = (
    df_latest_ais.join(
        df_latest_ports,
        on="mmsi",
        how="left"
    )
    .select(
        df_latest_ais["mmsi"],
        col("latitude"),
        col("longitude"),
        col("speed_over_ground"),
        col("vessel_name"),
        col("port_name").alias("destination_port"),
        col("eta")
    )
)

current_table_name = f"{catalog}.{gold_schema}.current_vessel_status"
df_gold_current.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(current_table_name)
print(f"Saved {current_table_name}")

# ==========================================
# 4. GOLD TABLE #2 — Vessel Trajectory (only changed positions)
# ==========================================
print("Building vessel_trajectory...")

window_spec_names = Window.partitionBy("mmsi").orderBy(col("call_timestamp").desc())
df_vessel_names = (
    df_ports.withColumn("rn", row_number().over(window_spec_names))
    .filter(col("rn") == 1)
    .select("mmsi", "vessel_name")
)

df_trajectory_raw = (
    df_ais.join(df_vessel_names, on="mmsi", how="left")
    .select(
        "mmsi",
        "vessel_name",
        "timestamp",
        "latitude",
        "longitude",
        "speed_over_ground",
        "course_over_ground",
        "heading"
    )
)

w = Window.partitionBy("mmsi").orderBy("timestamp")

df_gold_trajectory = (
    df_trajectory_raw
    .withColumn("prev_lat", lag("latitude").over(w))
    .withColumn("prev_lon", lag("longitude").over(w))
    .withColumn("prev_speed", lag("speed_over_ground").over(w))
    .filter(
        col("prev_lat").isNull()
        | (col("latitude") != col("prev_lat"))
        | (col("longitude") != col("prev_lon"))
        | (col("speed_over_ground") != col("prev_speed"))
    )
    .drop("prev_lat", "prev_lon", "prev_speed")
    .withColumn("recorded_at", date_format("timestamp", "yyyy-MM-dd HH:mm:ss"))
)

trajectory_table_name = f"{catalog}.{gold_schema}.vessel_trajectory"
df_gold_trajectory.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(trajectory_table_name)
print(f"Saved {trajectory_table_name}")

print("Gold layer build complete.")